# Getting Started with grasscalc

This notebook introduces the fundamental concepts and operations in the `grasscalc` package for working with Grassmannian manifolds.

## What is a Grassmannian?

The Grassmannian Gr(k, n) is the manifold of all k-dimensional subspaces of an n-dimensional vector space. Key facts:

- **Dimension**: dim Gr(k, n) = k(n - k)
- **Representation**: Points are represented as n×k orthonormal matrices
- **Equivalence**: U ~ UR for any orthogonal R (same column space)

In [ ]:
import numpy as np
from numpy.linalg import norm
np.set_printoptions(precision=4, suppress=True)

## 1. Sampling Points on the Grassmannian

The `sample_grassmann` function generates uniformly random points on Gr(k, n).

In [ ]:
from grasscalc.core.random import sample_grassmann

# Sample a random 3-dimensional subspace of R^7
rng = np.random.default_rng(42)
U = sample_grassmann(k=3, n=7, rng=rng)

print(f"U shape: {U.shape}")
print(f"\nU =\n{U}")

# Verify orthonormality: U^T U should be identity
print(f"\nU^T U (should be I_3):\n{U.T @ U}")

## 2. Distances on the Grassmannian

Several distance metrics are available:

- **Chordal distance**: Based on projector difference
- **Geodesic distance**: Riemannian distance (arc length)
- **Principal angles**: The canonical angles between subspaces

In [ ]:
from grasscalc.core.distances import (
    chordal_distance, geodesic_distance, principal_angles
)

# Sample two random subspaces
U = sample_grassmann(3, 7, rng)
V = sample_grassmann(3, 7, rng)

# Compute distances
d_chord = chordal_distance(U, V)
d_geo = geodesic_distance(U, V)
angles = principal_angles(U, V)

print(f"Chordal distance:  {d_chord:.4f}")
print(f"Geodesic distance: {d_geo:.4f}")
print(f"Principal angles:  {np.rad2deg(angles)} degrees")

## 3. Tangent Space Operations

The tangent space at U consists of matrices Xi satisfying U^T Xi = 0.

Key operations:
- **Projection**: Project any matrix to tangent space
- **Exponential map**: Move along geodesic
- **Logarithm map**: Inverse of exponential

In [ ]:
from grasscalc.core.tangent import (
    tangent_project, is_tangent, exponential_map, logarithm_map
)

# Create a tangent vector by projecting a random matrix
Z = rng.standard_normal((7, 3))
Xi = tangent_project(U, Z)

print(f"Is Xi tangent at U? {is_tangent(U, Xi)}")
print(f"U^T Xi (should be ≈ 0):\n{U.T @ Xi}")

In [ ]:
# Exponential map: move from U along direction Xi
Xi_unit = Xi / norm(Xi, 'fro')  # Unit tangent vector
t = 0.5  # Distance to move

V = exponential_map(U, t * Xi_unit)
print(f"V shape: {V.shape}")
print(f"V orthonormal? (V^T V = I): {np.allclose(V.T @ V, np.eye(3))}")

# Distance traveled should equal t
d = geodesic_distance(U, V)
print(f"Distance traveled: {d:.4f} (expected: {t})")

In [ ]:
# Logarithm map: find tangent vector from U to V
Xi_recovered = logarithm_map(U, V)
print(f"||Xi_recovered|| = {norm(Xi_recovered, 'fro'):.4f} (should equal distance {d:.4f})")

# Roundtrip: exp(log(U, V)) should give V
V_roundtrip = exponential_map(U, Xi_recovered)
print(f"Roundtrip matches? {np.allclose(V @ V.T, V_roundtrip @ V_roundtrip.T)}")

## 4. Geodesics

Geodesics are the shortest paths on the Grassmannian. They can be computed as curves parametrized by t ∈ [0, 1].

In [ ]:
from grasscalc.core.tangent import geodesic

# Geodesic from U to V
U = sample_grassmann(3, 7, rng)
V = sample_grassmann(3, 7, rng)

# Points along geodesic
t_values = [0.0, 0.25, 0.5, 0.75, 1.0]
points = [geodesic(U, V, t) for t in t_values]

# Verify distances are proportional to t
d_total = geodesic_distance(U, V)
print(f"Total distance: {d_total:.4f}")
print("\nt\t d(U, γ(t)) \t Expected")
print("-" * 40)
for t, W in zip(t_values, points):
    d = geodesic_distance(U, W)
    print(f"{t:.2f}\t {d:.4f}\t\t {t * d_total:.4f}")

## 5. Parallel Transport

Parallel transport moves tangent vectors along geodesics while preserving their geometric properties.

In [ ]:
from grasscalc.core.tangent import parallel_transport, tangent_inner_product

U = sample_grassmann(3, 7, rng)
V = sample_grassmann(3, 7, rng)

# Create two tangent vectors at U
Xi = tangent_project(U, rng.standard_normal((7, 3)))
Eta = tangent_project(U, rng.standard_normal((7, 3)))

# Transport both to V
Xi_V = parallel_transport(U, V, Xi)
Eta_V = parallel_transport(U, V, Eta)

# Verify properties are preserved
print("Property preservation under parallel transport:")
print(f"  ||Xi|| before: {norm(Xi, 'fro'):.4f}, after: {norm(Xi_V, 'fro'):.4f}")
print(f"  <Xi, Eta> before: {tangent_inner_product(U, Xi, Eta):.4f}")
print(f"  <Xi, Eta> after:  {tangent_inner_product(V, Xi_V, Eta_V):.4f}")
print(f"  Xi_V is tangent at V? {is_tangent(V, Xi_V)}")

## 6. The Sharp Bound Theorem

One of the main results in the grasscalc package is the Sharp Bound Theorem:

**Theorem**: For subspaces U ∈ Gr(k, N) and W ∈ Gr(k', N):
$$d^2(U, W) \geq |k - k'|$$

with equality if and only if one subspace contains the other.

In [ ]:
from grasscalc.layer2.sharp_bound import verify_sharp_bound

# Test the bound with random subspaces of different dimensions
N = 10
k, k_prime = 3, 5

for _ in range(5):
    U = sample_grassmann(k, N, rng)
    W = sample_grassmann(k_prime, N, rng)
    
    d_sq = chordal_distance(U, W) ** 2
    bound = abs(k - k_prime)
    
    print(f"d²(U, W) = {d_sq:.4f} >= {bound} = |k - k'| ✓" if d_sq >= bound - 1e-10 else "BOUND VIOLATED!")

In [ ]:
# Saturation case: when one subspace contains the other
U = sample_grassmann(3, 10, rng)

# Create W that contains U by extending with additional orthogonal vectors
# Use QR to get orthonormal complement
complement = np.eye(10) - U @ U.T
Q, R = np.linalg.qr(complement @ rng.standard_normal((10, 2)))
W = np.hstack([U, Q[:, :2]])  # W is 5-dim, contains U

d_sq = chordal_distance(U, W) ** 2
bound = abs(3 - 5)

print(f"Containment case: U (dim=3) ⊂ W (dim=5)")
print(f"d²(U, W) = {d_sq:.4f}")
print(f"|k - k'| = {bound}")
print(f"Saturation achieved: {np.isclose(d_sq, bound)}")

## Summary

This notebook covered the fundamental operations:

1. **Sampling**: Random points on Gr(k, n)
2. **Distances**: Chordal, geodesic, and principal angles
3. **Tangent space**: Projection, exp/log maps
4. **Geodesics**: Shortest paths on the manifold
5. **Parallel transport**: Moving tangent vectors
6. **Sharp bound**: Fundamental dimension inequality

Next notebooks cover optimization and the GCT chain for physics applications.